<div align="center">
  <img src="https://raw.githubusercontent.com/NaumanHSA/neurosurfer/main/docs/assets/banner/neurosurfer_banner_white.png" alt="Neurosurfer" width="45%"/>
</div>

<br/>

# 06 — Capstone: SQL Agent (ReAct vs Agentic Loop)

A **minimal, runnable test**: point a local model at a real SQL database through an MCP
server and check that both multi-step agent types actually drive it correctly.

This notebook builds:

1. A tiny synthetic shop database (`customers`, `products`, `orders`) — **SQLite** by default,
   or a **real MySQL** in Docker via a one-line `DB_BACKEND` switch (§3).
2. A minimal **read-only SQL MCP server** exposing `list_tables` / `describe_table` / `run_sql`
   — the same pattern used in [05 — Capstone: Insight Engine](05_capstone_insight_engine.ipynb),
   but self-contained here (no external support files). One script, either backend.
3. The **same question**, asked twice, against the **same tools**:
   - **`ReactAgent`** — the text-parsing Thought/Action/Observation loop ([01](01_providers_and_agents.ipynb) §5)
   - **`AgenticLoop`** — the native tool-calling loop ([01](01_providers_and_agents.ipynb) §4)
4. A **ground-truth check**: the expected numbers computed directly against the database, so you
   can eyeball whether each agent's answer is actually correct — not just "it ran".

**Why MCP instead of a hand-rolled SQL tool?** Neurosurfer already speaks MCP natively
(see [04 — MCP Servers](04_mcp_servers.ipynb)) — a `McpTool` slots into a `ToolPool` exactly
like a built-in tool, with zero framework code to write. There's no need to build a custom
`run_sql` `Tool` subclass (or reach for LangChain's SQL toolkit) when three `@server.tool()`
functions do the job.

**Model used:** any **native-tool-calling** local model works for both variants (`AgenticLoop`
needs it; `ReactAgent` doesn't but happily uses the same model — see [01](01_providers_and_agents.ipynb)
§5). Validated here with `qwen/qwen3.5-9b` via LM Studio on `localhost:1234` (the same model
validated for the SQL analyst node in [05](05_capstone_insight_engine.ipynb)).

---

## 1. Setup

Point Python at the repo root and confirm the `mcp` extra is installed:

```bash
pip install "neurosurfer[mcp]"
```

**Optional — run against a real MySQL.** To drive the agents against a real MySQL (e.g. in
Docker) instead of the throwaway SQLite file, install the driver and start a *throwaway*
container, then flip `DB_BACKEND` to `"mysql"` in §3:

```bash
pip install pymysql
docker run --name shop-mysql -p 3306:3306 \
  -e MYSQL_ROOT_PASSWORD=secret -e MYSQL_DATABASE=shop -d mysql:8
```

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os, sqlite3, random, tempfile
from pathlib import Path

# Point Python at the repo root when running from tutorials/
repo_root = Path(os.getcwd()).parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import neurosurfer
print(f"neurosurfer {neurosurfer.__version__}")

try:
    import mcp
    print("mcp SDK: OK")
except ImportError:
    print("mcp SDK missing — run: pip install 'neurosurfer[mcp]'")

# Gitignored scratch dir (tutorials/tmp/ already exists and is git-ignored).
TMP_DIR = repo_root / "tutorials" / "tmp"
TMP_DIR.mkdir(exist_ok=True)

In [ ]:
# ── LM Studio connection ──────────────────────────────────────────────────────
# Make sure LM Studio is running with a tool-calling model loaded and the local
# server ON (port 1234). AgenticLoop requires native tool calling; ReactAgent
# works with any model but we use the same one for a fair side-by-side.

LM_STUDIO_URL   = "http://localhost:1234/v1"
LM_STUDIO_MODEL = "qwen/qwen3.5-9b"   # adjust to your loaded model — needs tool_use capability
CONTEXT_WINDOW  = 48_000

from neurosurfer.llm.providers.openai import OpenAICompatProvider

provider = OpenAICompatProvider(
    model          = LM_STUDIO_MODEL,
    base_url       = LM_STUDIO_URL,
    api_key        = "lm-studio",
    context_window = CONTEXT_WINDOW,
)
print(f"Provider: {provider.model}")
print(f"Native tool calling: {provider.capabilities.tool_call_style}")

---

## 2. Approvals — no handler boilerplate

Some tools (shell, file writes, network) pause to ask a human before running. **Who**
answers is set by the agent's `approval` param:

- `approval="auto"` (the **default**) — approve everything, never block. Ideal for a
  notebook or an unattended script.
- `approval="ask"` — prompt right here (yes/no, or type an instruction to redirect the agent).
- `approval=my_handler` — plug in your own approval UI (this is how the CLI does it).

So agents **just run** — you never hand-write an `IOHandler`. (Our tools here are read-only
anyway, so nothing would prompt regardless.) The only place we name a handler explicitly is
the raw `ToolContext` in §5 — no agent wraps it — where we pass the ready-made
`AutoApproveIOHandler`, the same thing `approval="auto"` uses under the hood.

In [ ]:
from neurosurfer.tools import AutoApproveIOHandler

# Shipped with neurosurfer — no more hand-written approval handlers.
# Agents default to approval="auto"; this handler is only needed for the raw
# ToolContext in §5.
print("AutoApproveIOHandler ready (same as approval=\"auto\").")

## 3. Connect to the Database

This section connects the notebook to the **real database server** running remotely or inside your existing containerized environment.


In [ ]:
import os

import pyodbc

# ── Real database connection only (SQL Server) ────────────────────────────────
DB_CONFIG = dict(
    driver=os.getenv("DB_DRIVER", "ODBC Driver 17 for SQL Server"),
    server=os.getenv("DB_HOST", "localhost"),
    port=os.getenv("DB_PORT", "1433"),
    user=os.getenv("DB_USER", "sa"),
    password=os.getenv("DB_PASSWORD", "YourStrongPassword123!"),
    database=os.getenv("DB_NAME", "AIAccessManagment"),
)

# Parameter placeholder style for pyodbc.
PLACEHOLDER = "?"


def _conn_str():
    return (
        f"DRIVER={{{DB_CONFIG['driver']}}};"
        f"SERVER={DB_CONFIG['server']},{DB_CONFIG['port']};"
        f"DATABASE={DB_CONFIG['database']};"
        f"UID={DB_CONFIG['user']};"
        f"PWD={DB_CONFIG['password']};"
        f"TrustServerCertificate=yes;"
    )


def db_connect():
    """Return a pyodbc connection to the real SQL Server database."""
    return pyodbc.connect(_conn_str(), autocommit=True)


def db_test_connection():
    """
    Test the database connection.

    Returns:
        (True,  None)      — connection is healthy.
        (False, error_str) — connection failed.
    """
    try:
        con = db_connect()
        cur = con.cursor()
        cur.execute("SELECT 1")
        cur.fetchone()
        con.close()
        return True, None
    except Exception as exc:
        return False, str(exc)


def db_query(sql, params=()):
    """Run a query and return (columns, rows)."""
    con = db_connect()
    try:
        cur = con.cursor()
        cur.execute(sql, params)
        cols = [d[0] for d in cur.description] if cur.description else []
        return cols, cur.fetchall()
    finally:
        con.close()


# ── Verify the connection ─────────────────────────────────────────────────────
print("DB backend: SQL Server (pyodbc)")
print(f"  → {DB_CONFIG['user']}@{DB_CONFIG['server']},{DB_CONFIG['port']}/{DB_CONFIG['database']}")
print(f"  → driver: {DB_CONFIG['driver']}")

ok, err = db_test_connection()
if ok:
    print("  ✅ Connection verified — database is reachable.\n")
else:
    print(f"  ❌ Connection FAILED: {err}\n")

## 4. Test the Database Connection

With the `db_query` helper in place, this section runs a small set of validation
queries against the live database to confirm that:

- the connection opens and closes cleanly,
- simple `SELECT` queries return rows,
- aggregate queries (like `COUNT`) return expected scalar values,
- empty result sets are handled without error,
- and invalid queries fail gracefully with a meaningful message.

In [ ]:
# 1) Simple SELECT
cols, rows = db_query("SELECT TOP 5 CenterCode, CenterName, Region FROM core.Centers")
print(f"── SELECT centers: {len(rows)} rows | columns: {cols}")
for row in rows[:3]:
    print(f"    {row}")

# 2) Aggregation
cols, rows = db_query("SELECT COUNT(*) AS n FROM core.Users WHERE IsActive = 1")
print(f"\n── COUNT active users: {rows[0][0]}")

# 3) Parameterised query (note: ? placeholder for pyodbc)
cols, rows = db_query(
    "SELECT TOP 5 UserID, UserName FROM core.Users WHERE IsActive = ?",
    (1,),
)
print(f"\n── Parameterised lookup: {len(rows)} rows")
for row in rows:
    print(f"    {row}")

# 4) Empty result (valid)
cols, rows = db_query("SELECT * FROM core.Users WHERE UserID = -999999")
print(f"\n── Empty result: ok, 0 rows returned")

# 5) Invalid table (should fail)
try:
    db_query("SELECT * FROM core.NoSuchTable")
except Exception as exc:
    print(f"\n── Bad table rejected: {exc}")

---

## 4. A minimal read-only SQL MCP server

Three tools, exactly the pattern from [05](05_capstone_insight_engine.ipynb):
`list_tables`, `describe_table`, `run_sql` — the last one rejects anything that isn't a
read-only `SELECT`/`SHOW`/`DESCRIBE`/`EXPLAIN`, so the database is safe to hand to a local
model. The server reads its backend + connection info from **environment variables** (set in
§5), so the *same* script speaks either SQLite or MySQL. Written to a temp file and launched
as a **stdio** subprocess (same trick as [04](04_mcp_servers.ipynb) §3).

In [ ]:
SQL_MCP_SERVER_SRC = r'''
import os

from mcp.server.fastmcp import FastMCP

server = FastMCP("sql-agent-tutorial")

DB_DRIVER   = os.getenv("DB_DRIVER", "ODBC Driver 17 for SQL Server")
DB_SERVER   = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT     = os.getenv("DB_PORT", "1433")
DB_USER     = os.getenv("DB_USER", "sa")
DB_PASSWORD = os.getenv("DB_PASSWORD", "YourStrongPassword123!")
DB_NAME     = os.getenv("DB_NAME", "AIAccessManagment")


def _conn_str():
    return (
        f"DRIVER={{{DB_DRIVER}}};"
        f"SERVER={DB_SERVER},{DB_PORT};"
        f"DATABASE={DB_NAME};"
        f"UID={DB_USER};"
        f"PWD={DB_PASSWORD};"
        f"TrustServerCertificate=yes;"
    )


def _connect():
    import pyodbc
    return pyodbc.connect(_conn_str(), autocommit=True)


def _query(sql, params=(), cap=None):
    con = _connect()
    try:
        cur = con.cursor()
        cur.execute(sql, params)
        cols = [d[0] for d in cur.description] if cur.description else []
        rows = cur.fetchmany(cap) if cap else cur.fetchall()
        return cols, rows
    finally:
        con.close()


@server.tool(annotations={"readOnlyHint": True}, description="List every user table in the database, returned as schema-qualified names (e.g. core.Users).")
def list_tables() -> str:
    sql = (
        "SELECT TABLE_SCHEMA, TABLE_NAME "
        "FROM INFORMATION_SCHEMA.TABLES "
        "WHERE TABLE_TYPE = 'BASE TABLE' "
        "ORDER BY TABLE_SCHEMA, TABLE_NAME"
    )
    _, rows = _query(sql)
    return ", ".join(f"{r[0]}.{r[1]}" for r in rows) or "(no tables)"


@server.tool(annotations={"readOnlyHint": True}, description="Return the column names and types of a table. Accepts either a bare name (e.g. Users) or a schema-qualified name (e.g. core.Users).")
def describe_table(table: str) -> str:
    # Split schema.table if the caller passes a qualified name
    parts = table.split(".", 1)
    schema = parts[0] if len(parts) == 2 else None
    bare_name = parts[1] if len(parts) == 2 else parts[0]

    try:
        if schema:
            sql = (
                "SELECT COLUMN_NAME, DATA_TYPE "
                "FROM INFORMATION_SCHEMA.COLUMNS "
                "WHERE TABLE_SCHEMA = ? AND TABLE_NAME = ? "
                "ORDER BY ORDINAL_POSITION"
            )
            _, rows = _query(sql, (schema, bare_name))
        else:
            sql = (
                "SELECT COLUMN_NAME, DATA_TYPE "
                "FROM INFORMATION_SCHEMA.COLUMNS "
                "WHERE TABLE_NAME = ? "
                "ORDER BY ORDINAL_POSITION"
            )
            _, rows = _query(sql, (bare_name,))

        if not rows:
            return f"No such table: {table}"
        return "\n".join(f"{r[0]} {r[1]}" for r in rows)
    except Exception as e:
        return f"No such table: {table} ({e})"


@server.tool(annotations={"readOnlyHint": True}, description="Run a read-only SELECT query; returns up to 50 rows as text. Table names must be schema-qualified (e.g. core.Users, not just Users).")
def run_sql(query: str) -> str:
    q = query.strip().lower()
    if not q.startswith(("select", "with")):
        return "Only read-only queries (SELECT / WITH) are allowed."
    try:
        cols, rows = _query(query, cap=50)
        header = " | ".join(str(c) for c in cols)
        body = "\n".join(" | ".join(str(v) for v in r) for r in rows)
        return f"{header}\n{body}" if body else f"{header}\n(no rows)"
    except Exception as e:
        return f"SQL error: {e}"


if __name__ == "__main__":
    server.run("stdio")
'''


SERVER_PATH = str(TMP_DIR / "sql_mcp_server.py")
Path(SERVER_PATH).write_text(SQL_MCP_SERVER_SRC, encoding="utf-8")
print("wrote", SERVER_PATH, "| backend: sqlserver (pyodbc)")

---

## 5. Connect via `McpSession`

We want the connection to stay alive across **several cells** (both agent variants reuse it),
so `McpSession` — the synchronous wrapper from [05](05_capstone_insight_engine.ipynb) §5 — is
the right tool here, rather than `McpManager`'s `async with` block from
[04](04_mcp_servers.ipynb) (which tears the session down at the end of one cell/task).

In [ ]:
import sys
from neurosurfer.config.mcp import McpServerConfig
from neurosurfer.mcp import McpSession

# Pass connection info to the server subprocess via env vars — keeps the
# SA password out of process argv. The MCP SDK merges these onto a safe
# default environment (PATH etc.), so we only pass what the server needs.
server_env = {
    "DB_DRIVER":   DB_CONFIG["driver"],
    "DB_HOST":     DB_CONFIG["server"],
    "DB_PORT":     DB_CONFIG["port"],
    "DB_USER":     DB_CONFIG["user"],
    "DB_PASSWORD": DB_CONFIG["password"],
    "DB_NAME":     DB_CONFIG["database"],
}

session = McpSession([McpServerConfig(
    name="shop",
    transport="stdio",
    command=sys.executable,
    args=[SERVER_PATH],
    env=server_env,
)])

for s in session.connect():
    print(f"server {s.name!r}: connected={s.connected}, tools={s.tools}")

In [ ]:
from neurosurfer.tools import ToolPool, ToolContext, AutoApproveIOHandler
from neurosurfer.tools.builtin import FinishTool

pool = ToolPool([*session.tools(), FinishTool()])
ctx = ToolContext(cwd=repo_root, io=AutoApproveIOHandler())
print("pool:", pool.names())

---

## 6. The test question

The **same** question, **same** system prompt, **same** tool pool, for both variants below —
so any difference in the final answer comes from the agent loop, not from the setup. The
system prompt spells out one `run_sql` query per sub-question so a small local model doesn't
take a shortcut and guess a number instead of looking it up.

In [ ]:
from datetime import date

TODAY = date.today().isoformat()

SQL_SYSTEM_PROMPT = f"""\
ABOUT THIS ASSISTANT
You serve the AI Access Management system — a read-only, natural-language
assistant for authorised police operators. What you can do is limited to:
  - Answer questions about the system's data in plain language: police centers,
    operators (users), login sessions, inquiries/lookups, watchlisted persons,
    alerts, and alert rules — counts, lists, lookups, trends, rankings, and
    comparisons.
  - Generate PDF reports from that same data (for example, an operator activity
    summary for a given period).
You cannot modify data, perform administrative actions, or answer questions
about anything not held in this database.

DATABASE OVERVIEW
The database tracks access-management activity for a police operations network.
Key entities (schema prefix: core.):
  - Centers   — police centres / regional offices
  - Users      — operators (staff) who log in and run inquiries
  - Sessions   — login session records (who, when, duration, centre)
  - Inquiries  — lookups performed by operators (person, vehicle, etc.)
  - Persons    — watchlisted individuals (added by operators)
  - Alerts     — alert events and alert rules
Tables are namespaced under the `core` schema (e.g. core.Users, core.Persons).
Use list_tables and describe_table to discover exact table and column names
before writing any SQL — the schema above is a guide, not a substitute.

META
Today's date: {TODAY}
When a question says "today", "yesterday", "this week", "this month", or
"last week", resolve it relative to {TODAY}. A week runs Monday–Sunday.

DOMAIN NOTES (resolve loose wording to the schema before judging scope/ambiguity):
- Operators do NOT create or "add" other operator accounts. The only records an
  operator adds / enrolls / lists are watchlisted PERSONS
  (core.Persons.ListedByUserID = the operator who added that person). So "which
  operator added / listed / enrolled the most people / persons / users /
  individuals" is answerable over core.Persons — it is a normal data question,
  not out-of-scope or ambiguous.
- "users" / "operators" = the staff who log in and run inquiries (core.Users);
  "persons" / "people" on the watchlist = core.Persons. Use the surrounding
  wording to pick which one is meant.

GENERAL RULES
1. Always explore the schema first. Call list_tables to see what tables exist,
   then call describe_table on any table you plan to query — never assume column
   names, types, or relationships.
2. Discover the SQL dialect from errors. If a query fails, the error message
   will tell you what syntax is wrong. Adapt and retry.
3. Every number in your answer must come from a run_sql result. Never guess,
   estimate, or fabricate a value.
4. Break complex questions into sub-queries. Run them one at a time, verify each
   result before building the next query on it.
5. If a query returns a SQL error, immediately diagnose it (wrong column name?
   missing join? syntax?) and retry. Never give up on a sub-question after one
   failed attempt.
6. Use table-qualified column references in joins (e.g. table.column) to avoid
   ambiguity.
7. Always limit large result sets — use TOP, LIMIT, or FETCH FIRST depending on
   the dialect, so you never pull entire tables.
8. Only give your final answer once you have gathered all the data needed. If
   any sub-query is still unresolved, keep working.
9. Present your final answer clearly: state each finding with the concrete
   number from the query, then a brief interpretation.
"""


QUESTIONS = [
    "Which operator had the most inquiries last week?",
    "How many operators logged in today?",
    "Which center handled the most sessions this month?",
    "What are the top five operators by inquiry count?",
    "Which operator added the most watchlisted persons?",
    "How many watchlisted persons were added yesterday?",
    "Which center has the most watchlisted persons?",
    "What were the busiest login hours this week?",
    "How many inquiries were made today?",
]

print(f"Today: {TODAY}")
print(f"System prompt length: {len(SQL_SYSTEM_PROMPT)} chars")
print(f"Questions: {len(QUESTIONS)}")
for i, q in enumerate(QUESTIONS, 1):
    print(f"  {i}. {q}")

---

## 7. Variant A — `ReactAgent` (text-parsing ReAct)

Prompts the model to emit `Thought` / `Action` / `Action Input`, parses that text, executes
the tool, feeds back an `Observation`. No native tool-calling API required — this is the
loop to reach for on models that don't support (or poorly support) JSON function calling.
`verbose=False` so we render the trace ourselves, exactly as in [01](01_providers_and_agents.ipynb) §5.

In [ ]:
from neurosurfer.agents import ReactAgent, Guardrails, events
from neurosurfer.llm.types import GenerationConfig

react_agent = ReactAgent(
    provider      = provider,
    tools         = pool,
    system_prompt = SQL_SYSTEM_PROMPT,
    guardrails    = Guardrails(max_turns=16),
    cwd           = repo_root,
    gen_config    = GenerationConfig(max_tokens=2048, temperature=0.2),
    verbose       = False,
)   # approval defaults to "auto" — no IOHandler to write

question = QUESTIONS[0]
print(f"Task: {question}\n{'─' * 60}")

react_answer = ""
thinking = False

async for ev in react_agent.run(question):
    if isinstance(ev, events.ToolStarted):
        if thinking:
            print(); thinking = False
        print(f"🔧 {ev.title or ev.name}({ev.args})")
    elif isinstance(ev, events.ToolFinished):
        preview = ev.result.content[:150].replace("\n", " ")
        suffix = "…" if len(ev.result.content) > 150 else ""
        print(f"   ↳ {preview}{suffix}")
    elif isinstance(ev, events.ThinkingDelta):
        if not thinking:
            print("💭 Thinking…", end="", flush=True)
            thinking = True
    elif isinstance(ev, events.TextDelta):
        if thinking:
            print("\n"); thinking = False
        react_answer += ev.text
        print(ev.text, end="", flush=True)
    elif isinstance(ev, events.RunFinished):
        print(f"\n\n{'─' * 60}")
        print(f"Run finished — {ev.status}")
        react_answer = react_answer.strip() or ev.report or ""

---

## 8. Variant B — `AgenticLoop` (native tool calling)

Uses the provider's native function-calling API — no text parsing. Ends when the model calls
`finish` or reaches `max_turns`. This is the production default for models that support it
([01](01_providers_and_agents.ipynb) §4).

In [ ]:
from neurosurfer.agents import AgenticLoop

agentic_loop = AgenticLoop(
    provider      = provider,
    tools         = pool,
    system_prompt = SQL_SYSTEM_PROMPT + " Call finish with your findings once you're done.",
    guardrails    = Guardrails(max_turns=16),
    cwd           = repo_root,
    gen_config    = GenerationConfig(max_tokens=2048, temperature=0.2),
    verbose       = False,
)   # approval defaults to "auto" — no IOHandler to write

print(f"Task: {question}\n{'─' * 60}")

loop_answer = ""

async for ev in agentic_loop.run(question):
    if isinstance(ev, events.ToolStarted):
        print(f"🔧 {ev.name}({ev.args})")
    elif isinstance(ev, events.ToolFinished):
        preview = ev.result.content[:150].replace("\n", " ")
        suffix = "…" if len(ev.result.content) > 150 else ""
        print(f"   ↳ {preview}{suffix}")
    elif isinstance(ev, events.TextDelta):
        loop_answer += ev.text
        print(ev.text, end="", flush=True)
    elif isinstance(ev, events.RunFinished):
        print(f"\n\n{'─' * 60}")
        print(f"Run finished — {ev.status}")
        if ev.report:
            print(f"Report: {ev.report}")
        loop_answer = loop_answer.strip() or ev.report or ""

---

## 9. Cleanup

Disconnect from the MCP server (stops the subprocess and the background session thread).

In [ ]:
session.close()
print("MCP session closed.")

---

## Summary

| Step | API |
|---|---|
| Pick the backend | `DB_BACKEND = "sqlite"` / `"mysql"` → `db_connect()` / `db_query()` helpers |
| Describe the SQL server | `McpServerConfig(name, transport="stdio", command, args=[script], env=...)` |
| Keep it alive across cells | `McpSession(servers)` → `.connect()` / `.tools()` / `.close()` |
| Build the pool | `ToolPool([*session.tools(), FinishTool()])` |
| Text-parsing ReAct | `ReactAgent(provider, tools=pool, ...)` → `async for ev in agent.run(q)` |
| Native tool calling | `AgenticLoop(provider, tools=pool, ...)` → `async for ev in agent.run(q)` |

**Key ideas**

- MCP turns three plain Python functions into tools **any** agent type can call — `ReactAgent`
  and `AgenticLoop` see the exact same `ToolPool`, so this notebook isolates the *agent loop*
  as the only variable between the two runs.
- `ReactAgent` doesn't need native tool-calling support, but it's driven through the same
  permission-gated tool execution path as `AgenticLoop` — so a tool written once (or, as here,
  served once over MCP) works with either loop unmodified.
- A ground-truth query is cheap insurance: it tells you whether a wrong answer is a
  **tool-calling failure** (never got the right rows) or a **reasoning failure** (had the right
  rows, did the arithmetic wrong).

### What's next

- **[01 — Providers & Agents](01_providers_and_agents.ipynb):** `ReactAgent` vs `AgenticLoop` in depth.
- **[04 — MCP Servers](04_mcp_servers.ipynb):** connecting to real MCP servers, permissions, persistence.
- **[05 — Capstone: Insight Engine](05_capstone_insight_engine.ipynb):** the same SQL-over-MCP
  pattern as one node in a larger graph.
- **Make it yours:** the `DB_BACKEND = "mysql"` path already shows the pattern — swap the three
  `@server.tool()` bodies for Postgres (`psycopg`) or any other engine and the agent code doesn't
  change at all.